In [16]:
import os
os.environ["OMP_NUM_THREADS"] = "5"
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import KBinsDiscretizer



wq_df = pd.read_csv("datasets/winequality-red.csv", delimiter=";")
pd.set_option("display.max_rows", 20)
wq_df.tail(10)


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
1589,6.6,0.725,0.20,7.8,0.073,29.0,79.0,0.99770,3.29,0.54,9.2,5
1590,6.3,0.550,0.15,1.8,0.077,26.0,35.0,0.99314,3.32,0.82,11.6,6
1591,5.4,0.740,0.09,1.7,0.089,16.0,26.0,0.99402,3.67,0.56,11.6,6
1592,6.3,0.510,0.13,2.3,0.076,29.0,40.0,0.99574,3.42,0.75,11.0,6
1593,6.8,0.620,0.08,1.9,0.068,28.0,38.0,0.99651,3.42,0.82,9.5,6
1594,6.2,0.600,0.08,2.0,0.090,32.0,44.0,0.99490,3.45,0.58,10.5,5
1595,5.9,0.550,0.10,2.2,0.062,39.0,51.0,0.99512,3.52,0.76,11.2,6
1596,6.3,0.510,0.13,2.3,0.076,29.0,40.0,0.99574,3.42,0.75,11.0,6
1597,5.9,0.645,0.12,2.0,0.075,32.0,44.0,0.99547,3.57,0.71,10.2,5
1598,6.0,0.310,0.47,3.6,0.067,18.0,42.0,0.99549,3.39,0.66,11.0,6


In [17]:
y = wq_df.loc[:,"quality"]
X = wq_df.loc[:,:"alcohol"]
X

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol
0,7.4,0.700,0.00,1.9,0.076,11.0,34.0,0.99780,3.51,0.56,9.4
1,7.8,0.880,0.00,2.6,0.098,25.0,67.0,0.99680,3.20,0.68,9.8
2,7.8,0.760,0.04,2.3,0.092,15.0,54.0,0.99700,3.26,0.65,9.8
3,11.2,0.280,0.56,1.9,0.075,17.0,60.0,0.99800,3.16,0.58,9.8
4,7.4,0.700,0.00,1.9,0.076,11.0,34.0,0.99780,3.51,0.56,9.4
...,...,...,...,...,...,...,...,...,...,...,...
1594,6.2,0.600,0.08,2.0,0.090,32.0,44.0,0.99490,3.45,0.58,10.5
1595,5.9,0.550,0.10,2.2,0.062,39.0,51.0,0.99512,3.52,0.76,11.2
1596,6.3,0.510,0.13,2.3,0.076,29.0,40.0,0.99574,3.42,0.75,11.0
1597,5.9,0.645,0.12,2.0,0.075,32.0,44.0,0.99547,3.57,0.71,10.2


In [18]:

X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=.8, random_state=42)

In [20]:
model_test = LogisticRegression(max_iter=10000)
model_test.fit(X_train, y_train)
model_test.score(X_test, y_test)

0.571875

In [21]:
models = {"logreg": LogisticRegression(max_iter=10000), "svc": SVC(), "dt":DecisionTreeClassifier()}
precision_values = {"rounding": [6,5,4,3,2,1], "uniform binning": [2,3,4,5,6], "quantile binning": [2,3,4,5,6], "kmeans binning": [2,3,4,5,6]}
modes = ["both", "training_data_only", "testing_data_only"]
results = []
# huh = 0
for model_name, model in models.items():
    base_model = model.__class__(**model.get_params())
    base_model.fit(X_train, y_train)
    base_acc = base_model.score(X_test,y_test)

    results.append({"method":"base", "model": model_name, "level": 0, "mode": "base", "accuracy": base_acc})

    for method, levels in precision_values.items():
        for level in levels:
            for mode in modes:

                X_train_new = X_train.copy()
                X_test_new = X_test.copy()

                if method == "rounding":
                    if mode == "both":
                        X_train_new = X_train_new.round(level)
                        X_test_new = X_test_new.round(level)
                    elif mode == "training_data_only":
                        X_train_new = X_train_new.round(level)
                    elif mode == "testing_data_only":
                        X_test_new = X_test_new.round(level)

                elif method == "uniform binning":
                    kbd = KBinsDiscretizer(n_bins=level, encode="ordinal", strategy="uniform")
                    kbd.fit(X_train_new)
                    if mode == "both":
                        X_train_new = pd.DataFrame(kbd.transform(X_train_new), columns=X_train.columns)
                        X_test_new = pd.DataFrame(kbd.transform(X_test_new), columns=X_test.columns)
                    elif mode == "training_data_only":
                        X_train_new = pd.DataFrame(kbd.transform(X_train_new), columns=X_train.columns)
                    elif mode == "testing_data_only":
                        X_test_new = pd.DataFrame(kbd.transform(X_test_new), columns=X_test.columns)
                
                elif method == "quantile binning":
                    kbd = KBinsDiscretizer(n_bins=level, encode="ordinal", strategy="quantile")
                    kbd.fit(X_train_new)
                    if mode == "both":
                        X_train_new = pd.DataFrame(kbd.transform(X_train_new), columns=X_train.columns)
                        X_test_new = pd.DataFrame(kbd.transform(X_test_new), columns=X_test.columns)
                    elif mode == "training_data_only":
                        X_train_new = pd.DataFrame(kbd.transform(X_train_new), columns=X_train.columns)
                    elif mode == "testing_data_only":
                        X_test_new = pd.DataFrame(kbd.transform(X_test_new), columns=X_test.columns)
                
                elif method == "kmeans binning":
                    kbd = KBinsDiscretizer(n_bins=level, encode="ordinal", strategy="kmeans")
                    kbd.fit(X_train_new)
                    if mode == "both":
                        X_train_new = pd.DataFrame(kbd.transform(X_train_new), columns=X_train.columns)
                        X_test_new = pd.DataFrame(kbd.transform(X_test_new), columns=X_test.columns)
                    elif mode == "training_data_only":
                        X_train_new = pd.DataFrame(kbd.transform(X_train_new), columns=X_train.columns)
                    elif mode == "testing_data_only":
                        X_test_new = pd.DataFrame(kbd.transform(X_test_new), columns=X_test.columns)
                

                exp_model = model.__class__(**model.get_params())
                exp_model.fit(X_train_new, y_train)

                exp_acc = exp_model.score(X_test_new, y_test)
                # if method=="quantile binning" and level==2 and mode == "both":
                #     huh += 1

                
                results.append({"method": method, "model":model_name, "level": level, "mode": mode, "accuracy":exp_acc})


c:\Users\abelg\anaconda3\envs\ress\Lib\site-packages\sklearn\preprocessing\_discretization.py:296: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(
c:\Users\abelg\anaconda3\envs\ress\Lib\site-packages\sklearn\preprocessing\_discretization.py:296: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(
c:\Users\abelg\anaconda3\envs\ress\Lib\site-packages\sklearn\preprocessing\_discretization.py:296: FutureWarning: The current default behavior, quanti

In [19]:
pd.set_option("display.max_rows", None)
results = pd.DataFrame(results)
results

,method,model,level,mode,accuracy
0,base,logreg,0,base,0.571875
1,rounding,logreg,6,both,0.571875
2,rounding,logreg,6,training_data_only,0.571875
3,rounding,logreg,6,testing_data_only,0.571875
4,rounding,logreg,5,both,0.571875
5,rounding,logreg,5,training_data_only,0.571875
6,rounding,logreg,5,testing_data_only,0.571875
7,rounding,logreg,4,both,0.571875
8,rounding,logreg,4,training_data_only,0.571875
9,rounding,logreg,4,testing_data_only,0.571875


In [32]:
data = pd.Series([2.3, 5.1, 7.8, 1.2, 9.0, 4.5, 6.7, 3.3])

binned = pd.cut(data, bins=3)

print(binned)

0    (1.192, 3.8]
1      (3.8, 6.4]
2      (6.4, 9.0]
3    (1.192, 3.8]
4      (6.4, 9.0]
5      (3.8, 6.4]
6      (6.4, 9.0]
7    (1.192, 3.8]
dtype: category
Categories (3, interval[float64, right]): [(1.192, 3.8] < (3.8, 6.4] < (6.4, 9.0]]


In [33]:
X_thing = np.array([[2.3],
              [5.1],
              [7.8],
              [1.2],
              [9.0],
              [4.5]])

kbd = KBinsDiscretizer(
    n_bins=3,
    encode='ordinal',      
    strategy='uniform'     
)

X_binned = kbd.fit_transform(X_thing)

print("Binned values:")
print(X_binned)

print("Bin edges:")
print(kbd.bin_edges_)

Binned values:
[[0.]
 [1.]
 [2.]
 [0.]
 [2.]
 [1.]]
Bin edges:
[array([1.2, 3.8, 6.4, 9. ])]


In [34]:
kbd = KBinsDiscretizer(n_bins=5, encode="ordinal", strategy="uniform")

binned = kbd.fit_transform(X)

binned_df = pd.DataFrame(binned, columns=X.columns)
binned_df

,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,fractal_dimension_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,2.0,0.0,2.0,1.0,2.0,3.0,3.0,3.0,3.0,3.0,...,3.0,0.0,3.0,2.0,3.0,3.0,2.0,4.0,2.0,2.0
1,3.0,1.0,3.0,2.0,1.0,0.0,1.0,1.0,1.0,0.0,...,3.0,1.0,2.0,2.0,1.0,0.0,0.0,3.0,1.0,1.0
2,3.0,1.0,2.0,2.0,2.0,2.0,2.0,3.0,2.0,1.0,...,2.0,1.0,2.0,1.0,2.0,1.0,1.0,4.0,2.0,1.0
3,1.0,1.0,1.0,0.0,4.0,4.0,2.0,2.0,3.0,4.0,...,1.0,1.0,1.0,0.0,4.0,4.0,2.0,4.0,4.0,3.0
4,3.0,0.0,3.0,2.0,2.0,1.0,2.0,2.0,1.0,0.0,...,2.0,0.0,2.0,1.0,2.0,0.0,1.0,2.0,0.0,0.0
5,1.0,1.0,1.0,0.0,3.0,2.0,1.0,2.0,2.0,2.0,...,1.0,1.0,1.0,0.0,3.0,2.0,2.0,2.0,2.0,2.0
6,2.0,1.0,2.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,...,2.0,2.0,2.0,1.0,2.0,1.0,1.0,3.0,1.0,0.0
7,1.0,1.0,1.0,0.0,2.0,2.0,1.0,1.0,2.0,2.0,...,1.0,2.0,1.0,0.0,3.0,1.0,1.0,2.0,1.0,1.0
8,1.0,2.0,1.0,0.0,3.0,2.0,2.0,2.0,3.0,2.0,...,1.0,2.0,1.0,0.0,3.0,2.0,2.0,3.0,2.0,1.0
9,1.0,2.0,1.0,0.0,2.0,3.0,2.0,2.0,2.0,3.0,...,1.0,3.0,1.0,0.0,3.0,4.0,4.0,3.0,2.0,4.0


In [35]:
kbd.bin_edges_

array([array([ 6.981 , 11.2068, 15.4326, 19.6584, 23.8842, 28.11  ]),
       array([ 9.71 , 15.624, 21.538, 27.452, 33.366, 39.28 ]),
       array([ 43.79 ,  72.732, 101.674, 130.616, 159.558, 188.5  ]),
       array([ 143.5,  615. , 1086.5, 1558. , 2029.5, 2501. ]),
       array([0.05263 , 0.074784, 0.096938, 0.119092, 0.141246, 0.1634  ]),
       array([0.01938 , 0.084584, 0.149788, 0.214992, 0.280196, 0.3454  ]),
       array([0.     , 0.08536, 0.17072, 0.25608, 0.34144, 0.4268 ]),
       array([0.     , 0.04024, 0.08048, 0.12072, 0.16096, 0.2012 ]),
       array([0.106 , 0.1456, 0.1852, 0.2248, 0.2644, 0.304 ]),
       array([0.04996 , 0.059456, 0.068952, 0.078448, 0.087944, 0.09744 ]),
       array([0.1115, 0.6638, 1.2161, 1.7684, 2.3207, 2.873 ]),
       array([0.3602 , 1.26516, 2.17012, 3.07508, 3.98004, 4.885  ]),
       array([ 0.757 ,  5.0016,  9.2462, 13.4908, 17.7354, 21.98  ]),
       array([  6.802 , 113.8816, 220.9612, 328.0408, 435.1204, 542.2   ]),
       array([0.0017

NameError: name 'num_df' is not defined

In [ ]:
X_train_binned = kbd.fit_transform(X_train)

for i in range(X_train_binned.shape[1]):
    edges = kbd.bin_edges_[i]
    mids = (edges[:-1] + edges[1:]) / 2
    X_train_binned[:, i] = mids[X_train_binned[:, i].astype(int)]
X_train_binned

array([[1.279575e+01, 1.710250e+01, 8.306500e+01, ..., 2.182500e-01,
        5.369750e-01, 1.435100e-01],
       [2.300525e+01, 3.188750e+01, 1.533550e+02, ..., 2.182500e-01,
        2.833250e-01, 1.435100e-01],
       [1.279575e+01, 1.710250e+01, 8.306500e+01, ..., 7.275000e-02,
        2.833250e-01, 8.453000e-02],
       ...,
       [1.279575e+01, 1.710250e+01, 8.306500e+01, ..., 7.275000e-02,
        2.833250e-01, 8.453000e-02],
       [1.279575e+01, 1.710250e+01, 8.306500e+01, ..., 2.182500e-01,
        2.833250e-01, 8.453000e-02],
       [1.279575e+01, 1.710250e+01, 8.306500e+01, ..., 7.275000e-02,
        2.833250e-01, 8.453000e-02]], shape=(455, 30))

In [48]:
df = pd.DataFrame({
    "feature1": [10, 12, 14, 16, 18],
    "feature2": [-9, 3, 5, 7, 9]
})
kbdddd = KBinsDiscretizer(n_bins=2, encode="ordinal", strategy="uniform")
newww = kbdddd.fit_transform(df)
kbdddd.bin_edges_

array([array([10., 14., 18.]), array([-9.,  0.,  9.])], dtype=object)